# Modelado xG: Regresión Lineal

En este notebook entrenaremos un modelo de Regresión Lineal para predecir la probabilidad de gol (xG) de cada tiro, buscando superar el benchmark del 88% de acierto.

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

# Cargar datos procesados
df = pd.read_csv('../data/processed/shots_features.csv')

# Limpieza básica: Eliminar filas con NaNs en las features críticas
features = ['distance', 'angle', 'is_header', 'is_big_chance', 'is_penalty', 'is_counter']
df = df.dropna(subset=features + ['is_goal'])

X = df[features]
y = df['is_goal'].astype(int)

print(f"Tiros totales para modelar: {len(df)}")

Tiros totales para modelar: 7198


### 1. División del Dataset (Train, Test, Retest)
Dividiremos los datos en 60% entrenamiento, 20% prueba y 20% re-test (validación final).

In [6]:
# Primero separamos el 60% para train
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)

# El 40% restante lo dividimos a la mitad (20% test, 20% retest)
X_test, X_retest, y_test, y_retest = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {len(X_train)} | Test: {len(X_test)} | Retest: {len(X_retest)}")

Train: 4318 | Test: 1440 | Retest: 1440


### 2. Entrenamiento del Modelo

In [7]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Modelo entrenado exitosamente.")

Modelo entrenado exitosamente.


### 3. Evaluación y Comparación con Benchmark (88%)

In [8]:
def evaluate(X_data, y_true, label):
    y_pred = model.predict(X_data)
    # Convertir probabilidad continua a binaria (umbral 0.5)
    y_pred_bin = [1 if p > 0.5 else 0 for p in y_pred]
    
    acc = accuracy_score(y_true, y_pred_bin)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    print(f"--- Métricas {label} ---")
    print(f"Accuracy: {acc:.4f} (Benchmark: 0.8800)")
    print(f"RMSE: {rmse:.4f}")
    return acc

acc_test = evaluate(X_test, y_test, "Test")
acc_retest = evaluate(X_retest, y_retest, "Retest")

--- Métricas Test ---
Accuracy: 0.8965 (Benchmark: 0.8800)
RMSE: 0.2845
--- Métricas Retest ---
Accuracy: 0.8875 (Benchmark: 0.8800)
RMSE: 0.2974
